# Figure 2 -- OmegAMP conditioning

In [ ]:
import pandas as pd, numpy as np, re, warnings
from pathlib import Path
from Bio import SeqIO
from scipy.stats import binned_statistic_2d, gaussian_kde
import matplotlib.pyplot as plt, matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle, Patch
import modlamp.analysis as manalysis
warnings.filterwarnings('ignore')

plt.style.use('default')
plt.rcParams.update({
    'font.size':7, 'axes.titlesize':7, 'axes.labelsize':6,
    'xtick.labelsize':6, 'ytick.labelsize':6, 'legend.fontsize':6,
    'axes.linewidth':0.5, 'axes.edgecolor':'grey',
    'xtick.color':'#555555', 'ytick.color':'#555555',
    'axes.labelcolor':'#333333', 'font.family':'sans-serif',
    'font.sans-serif':['Arial','DejaVu Sans'],
    'figure.dpi':300, 'savefig.dpi':300,
    'xtick.major.width':0.6, 'ytick.major.width':0.6,
    'xtick.major.size':2, 'ytick.major.size':2,
    'axes.spines.top':True, 'axes.spines.right':True,
    'figure.facecolor':'white', 'axes.facecolor':'white',
})

ROOT = Path("../data")
SPECIES = ROOT / "figure2/species"
TRAINING = ROOT / "figure2/training/generative-model-dataset.csv"
TARGETED = ROOT / "figure2/conditioning"
APEX = ROOT / "figure2/apex"
BENCH = ROOT / "figure2/benchmarks"

AGGRESCAN = {'I':1.822,'F':1.754,'V':1.594,'L':1.38,'Y':1.159,'W':1.037,'M':0.91,'C':0.604,
             'A':-0.036,'T':-0.159,'S':-0.294,'P':-0.334,'G':-0.535,'K':-0.931,'H':-1.033,
             'Q':-1.231,'R':-1.24,'N':-1.302,'E':-1.412,'D':-1.836}
AA_MW = {'A':89.09,'R':174.20,'N':132.12,'D':133.10,'C':121.16,'E':147.13,'Q':146.15,
         'G':75.03,'H':155.16,'I':131.17,'L':131.17,'K':146.19,'M':149.21,'F':165.19,
         'P':115.13,'S':105.09,'T':119.12,'W':204.23,'Y':181.19,'V':117.15}
WATER = 18.015
def peptide_mw(seq): return sum(AA_MW.get(aa,0) for aa in str(seq).upper()) - (len(str(seq))-1)*WATER
def modlamp_charge(seqs): h=manalysis.GlobalAnalysis(list(seqs)); h.calc_charge(); return h.charge[0]
def modlamp_hydro(seqs): h=manalysis.GlobalAnalysis(list(seqs)); h.calc_H(scale='eisenberg'); return h.H[0]
def seq_agg(s):
    sc = [AGGRESCAN.get(aa,0) for aa in str(s).upper() if aa in AGGRESCAN]
    if len(sc)<5: return np.mean(sc) if sc else np.nan
    return np.mean([np.mean(sc[i:i+5]) for i in range(len(sc)-4)])
def calc_props(seqs):
    s=list(seqs); return pd.DataFrame({'charge':modlamp_charge(s),'length':[len(x) for x in s],'hydrophobicity':modlamp_hydro(s)})

# -- DATA LOADING (unchanged) --
print("Panel A: species data...")
sp_dfs = [pd.read_csv(f) for f in SPECIES.glob("*.csv")]
sp_df = pd.concat(sp_dfs, ignore_index=True)
sp_agg = sp_df.groupby('sequence').agg(activity=('activity','mean')).reset_index()
sp_agg['mw'] = sp_agg['sequence'].apply(peptide_mw)
sp_agg['activity_um'] = sp_agg['activity'] / (sp_agg['mw']/1000)
sp_agg['aggregation'] = sp_agg['sequence'].apply(seq_agg)
sp_agg['charge'] = modlamp_charge(sp_agg['sequence'].tolist())
sp_agg['hydrophobicity'] = modlamp_hydro(sp_agg['sequence'].tolist())
print(f"  {len(sp_agg)} seqs")

print("Panel B: conditioning...")
single = TARGETED/"single-property"; multi = TARGETED/"multi-property"
multi_conds = {'low_charge_short':{'charge':2,'length':10,'hydrophobicity':-0.2},
               'moderate_low':{'charge':4,'length':15,'hydrophobicity':0.0},
               'moderate':{'charge':6,'length':20,'hydrophobicity':0.3},
               'high_moderate':{'charge':8,'length':25,'hydrophobicity':0.4},
               'high_long':{'charge':10,'length':30,'hydrophobicity':0.6}}
raw_cond = {'charge':[],'length':[],'hydrophobicity':[]}
agg_cond = {'charge':[],'length':[],'hydrophobicity':[]}
for ct in [0,2,4,6,8,10]:
    fp = single/f"charge-{ct}"/"samples.fasta"
    if fp.exists():
        seqs=[str(r.seq) for r in SeqIO.parse(fp,'fasta')]; vals=modlamp_charge(seqs)
        for v in vals: raw_cond['charge'].append({'target':ct,'achieved':v,'cond':'C'})
        agg_cond['charge'].append({'target':ct,'mean':np.mean(vals),'std':np.std(vals),'cond':'C'})
for lt in [10,15,20,25,30]:
    fp = single/f"length-{lt}"/"samples.fasta"
    if fp.exists():
        seqs=[str(r.seq) for r in SeqIO.parse(fp,'fasta')]; vals=np.array([len(s) for s in seqs])
        for v in vals: raw_cond['length'].append({'target':lt,'achieved':v,'cond':'L'})
        agg_cond['length'].append({'target':lt,'mean':np.mean(vals),'std':np.std(vals),'cond':'L'})
for ht in [-0.5,-0.2,0.0,0.2,0.4,0.6,0.8]:
    hc=str(ht).replace('.','p').replace('-','m')
    fp = single/f"hydrophob-{hc}"/"samples.fasta"
    if fp.exists():
        seqs=[str(r.seq) for r in SeqIO.parse(fp,'fasta')]; vals=modlamp_hydro(seqs)
        for v in vals: raw_cond['hydrophobicity'].append({'target':ht,'achieved':v,'cond':'H'})
        agg_cond['hydrophobicity'].append({'target':ht,'mean':np.mean(vals),'std':np.std(vals),'cond':'H'})
for dn, targs in multi_conds.items():
    fp = multi/dn/"samples.fasta"
    if fp.exists():
        seqs=[str(r.seq) for r in SeqIO.parse(fp,'fasta')]; props=calc_props(seqs)
        for prop in ['charge','length','hydrophobicity']:
            for v in props[prop].values: raw_cond[prop].append({'target':targs[prop],'achieved':v,'cond':'C+H+L'})
            agg_cond[prop].append({'target':targs[prop],'mean':props[prop].mean(),'std':props[prop].std(),'cond':'C+H+L'})
for p in raw_cond: raw_cond[p]=pd.DataFrame(raw_cond[p]); agg_cond[p]=pd.DataFrame(agg_cond[p])

print("Panel C: grid sweep...")
charge_grid=[0,2,4,6,8,10,12]; hydro_grid=[-0.75,-0.50,-0.25,0.0,0.25,0.5,0.75]
def fmt_h(h):
    m={-0.75:"-0.75",-0.50:"-0.50",-0.25:"-0.25",0.0:"0.0",0.25:"0.25",0.5:"0.5",0.75:"0.75"}
    return m.get(h,str(h)).replace('.','p').replace('-','m')
grid_data=[]
for ct in charge_grid:
    for ht in hydro_grid:
        dn=f"c{ct}_h{fmt_h(ht)}"
        fp=TARGETED/"grid-sweep-charge-hydrophob"/dn/"samples.fasta"
        apex_p=APEX/"grid-sweep-charge-hydrophob"/f"{dn}-apex-predictions-min.tsv"
        if not fp.exists(): continue
        seqs=[str(r.seq) for r in SeqIO.parse(fp,'fasta')]; props=calc_props(seqs)
        ok = (np.abs(props['charge']-ct)<=0.7)&(np.abs(props['hydrophobicity']-ht)<=0.1)
        apex_mic=np.nan
        if apex_p.exists(): adf=pd.read_csv(apex_p,sep='\t'); apex_mic=np.log2(adf.groupby('Sequence_id')['MIC'].min().mean())
        grid_data.append({'ct':ct,'ht':ht,'charge_actual':props['charge'].mean(),'hydrophob_actual':props['hydrophobicity'].mean(),
                         'success':ok.mean(),'apex':apex_mic,'agg':np.mean([seq_agg(s) for s in seqs])})
grid_df=pd.DataFrame(grid_data)

print("Panel D: benchmarking...")
models={'amp-gan.fasta':'AMP-GAN','amp-diffusion.fasta':'AMP-Diffusion','diff-amp.fasta':'Diff-AMP',
    'cpldiff.fasta':'CPL-Diff','hydramp.fasta':'HydrAMP','amp_lm_MAX40.fasta':'AMP-LM',
    'amp_muller_MAX40.fasta':'AMP-Muller','unconditional.fasta':'OmegAMP-U',
    'target-physicochemical.fasta':'OmegAMP-T','population-guided.fasta':'OmegAMP-P'}
crit={'charge':(2,10),'length':(5,30),'hydrophobicity':(-0.5,0.8)}
bench_results={}
for fn,name in models.items():
    fp=BENCH/fn
    if not fp.exists(): continue
    seqs=[str(r.seq) for r in SeqIO.parse(fp,'fasta')]
    if not seqs: continue
    print(f"  {name}...",end=' ')
    c=modlamp_charge(seqs); l=np.array([len(s) for s in seqs]); h=modlamp_hydro(seqs)
    gc=(c>=crit['charge'][0])&(c<=crit['charge'][1]); gl=(l>=crit['length'][0])&(l<=crit['length'][1])
    gh=(h>=crit['hydrophobicity'][0])&(h<=crit['hydrophobicity'][1])
    bench_results[name]={'C':gc.mean()*100,'L':gl.mean()*100,'H':gh.mean()*100,
        'C+L':(gc&gl).mean()*100,'C+H':(gc&gh).mean()*100,'L+H':(gl&gh).mean()*100,
        'C+L+H':(gc&gl&gh).mean()*100,'n':len(seqs)}
    print(f"{bench_results[name]['C+L+H']:.1f}%")

print("Panel E: distributions...")
training=pd.read_csv(TRAINING)
amp_seqs=training[training['IsAMP']==1]['Sequence'].tolist()
pep_seqs=training[training['IsAMP']==0]['Sequence'].tolist()
print(f"  AMPs..."); amp_props=calc_props(amp_seqs)
print(f"  Peptipedia..."); pep_props=calc_props(pep_seqs)
unc_seqs=[str(r.seq) for r in SeqIO.parse(BENCH/"unconditional.fasta",'fasta')]
tar_seqs=[str(r.seq) for r in SeqIO.parse(BENCH/"target-physicochemical.fasta",'fasta')]
pop_seqs=[str(r.seq) for r in SeqIO.parse(BENCH/"population-guided.fasta",'fasta')]
print(f"  OmegAMP modes..."); unc_props=calc_props(unc_seqs); tar_props=calc_props(tar_seqs); pop_props=calc_props(pop_seqs)
print("\nPlotting...")

# ================================================================
# FIGURE
# ================================================================
fig = plt.figure(figsize=(6.5, 8), dpi=300)
gs_a = gridspec.GridSpec(1,2,figure=fig,left=0.10,right=0.97,top=0.97,bottom=0.84,wspace=0.35)
gs_b = gridspec.GridSpec(1,3,figure=fig,left=0.08,right=0.97,top=0.80,bottom=0.67,wspace=0.40)
gs_c = gridspec.GridSpec(1,3,figure=fig,left=0.06,right=0.97,top=0.62,bottom=0.45,wspace=0.50)
gs_d = gridspec.GridSpec(3,1,figure=fig,left=0.10,right=0.50,top=0.40,bottom=0.20,hspace=0.1,height_ratios=[3,1,2])
gs_e = gridspec.GridSpec(1,2,figure=fig,left=0.60,right=0.97,top=0.40,bottom=0.24,wspace=0.08)
PL = dict(fontsize=10,fontweight='bold',va='top',ha='left')

from matplotlib.colors import TwoSlopeNorm
from mpl_toolkits.axes_grid1 import make_axes_locatable

# -- A: MIC + Aggregation heatmaps --
for pi in range(2):
    ax=fig.add_subplot(gs_a[pi])
    if pi==0: ax.text(-0.08,1.12,'A',transform=ax.transAxes,**PL)
    x,y=sp_agg['charge'].values,sp_agg['hydrophobicity'].values
    if pi==0:
        z_vals=np.log2(sp_agg['activity_um'].clip(1)).values
        cmap='RdBu'; norm=TwoSlopeNorm(vcenter=5,vmin=0,vmax=7); label=r'log$_2$ MIC ($\mu$M)'
    else:
        z_vals=sp_agg['aggregation'].values
        cmap='RdYlBu_r'; norm=TwoSlopeNorm(vcenter=0,vmin=-0.5,vmax=1.0); label='Aggregation Propensity'
    stat,xe,ye,_=binned_statistic_2d(x,y,z_vals,statistic='mean',bins=[20,20],range=[[0,30],[-2.5,1.0]])
    im=ax.imshow(stat.T,origin='lower',aspect='auto',cmap=cmap,norm=norm,extent=[xe[0],xe[-1],ye[0],ye[-1]],interpolation='nearest')
    ax.set_xlabel('Charge',labelpad=1,fontsize=6); ax.set_ylabel('Hydrophobicity',labelpad=1,fontsize=6)
    ax.tick_params(axis='both',length=2,width=0.6,labelsize=6)
    for spine in ax.spines.values(): spine.set_edgecolor('grey'); spine.set_linewidth(0.5)
    cb=plt.colorbar(im,ax=ax,shrink=0.9,pad=0.02); cb.set_label(label,fontsize=6,labelpad=2); cb.ax.tick_params(labelsize=6,width=0.5,length=2)

# -- B: Conditioning accuracy --
cc={'C':'#0173B2','L':'#DE8F05','H':'#029E73','C+H+L':'#CC78BC'}
for pi,prop in enumerate(['charge','length','hydrophobicity']):
    ax=fig.add_subplot(gs_b[pi])
    if pi==0: ax.text(-0.12,1.12,'B',transform=ax.transAxes,**PL)
    rels=['C','C+H+L'] if prop=='charge' else ['L','C+H+L'] if prop=='length' else ['H','C+H+L']
    xl={'charge':'Target C','length':'Target L','hydrophobicity':'Target H'}[prop]
    yl={'charge':'Achieved C','length':'Achieved L','hydrophobicity':'Achieved H'}[prop]
    rmse_txts=[]
    for cond in rels:
        sub_r=raw_cond[prop][raw_cond[prop]['cond']==cond]
        ax.scatter(sub_r['target'],sub_r['achieved'],alpha=0.15,s=6,color=cc[cond],edgecolors='none',zorder=5,rasterized=True)
        sub_a=agg_cond[prop][agg_cond[prop]['cond']==cond].sort_values('target')
        if len(sub_a)>0:
            ax.errorbar(sub_a['target'],sub_a['mean'],yerr=sub_a['std'],fmt='o',capsize=2,
                       color=cc[cond],ms=3,lw=1,elinewidth=0.8,capthick=0.5,markeredgecolor='white',markeredgewidth=0.4,zorder=10,label=cond)
            rmse_txts.append((cond,np.sqrt(np.mean((sub_a['target']-sub_a['mean'])**2)),cc[cond]))
    lims=[raw_cond[prop]['target'].min(),raw_cond[prop]['target'].max()]
    ax.plot(lims,lims,'--',color='#999',lw=0.7,zorder=1)
    by=0.18 if prop!='length' else 0.85
    ax.text(0.97,by+0.11,'RMSE:',transform=ax.transAxes,ha='right',fontsize=6,color='#555')
    for i,(c,r,col) in enumerate(rmse_txts):
        ax.text(0.97,by-i*0.11,f'{c}: {r:.2f}',transform=ax.transAxes,ha='right',fontsize=6,color=col)
    ax.set_xlabel(xl); ax.set_ylabel(yl)
    ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.legend(frameon=False,fontsize=6,loc='upper left',handlelength=1,markerscale=0.8)

# -- C: Grid sweep (circles at actual coords, X at targets, matching original) --
x_marker_colors = ['darkred', 'orange', 'darkgreen']
grid_cmaps = ['RdYlGn', 'RdBu', 'RdYlBu_r']
grid_labels = [r'Conditioning Accuracy (%)', r'log$_2$ $MIC_{APEX}$ ($\mu$M)', 'Aggregation Propensity']
grid_vmins = [0, 0, -0.5]
grid_vmaxs = [1, 7, 1.5]
grid_cols = ['success', 'apex', 'agg']

for pi in range(3):
    ax=fig.add_subplot(gs_c[pi])
    if pi==0: ax.text(-0.12,1.12,'C',transform=ax.transAxes,**PL)
    # Training data density cloud
    ac=amp_props['charge'].values; ah=amp_props['hydrophobicity'].values
    valid=~np.isnan(ac)&~np.isnan(ah)
    try:
        kde=gaussian_kde(np.vstack([ac[valid],ah[valid]]),bw_method='scott')
        cg=np.linspace(ac[valid].min()-1,ac[valid].max()+1,100)
        hg=np.linspace(ah[valid].min()-0.2,ah[valid].max()+0.2,100)
        Cg,Hg=np.meshgrid(cg,hg)
        Zg=kde(np.vstack([Cg.ravel(),Hg.ravel()])).reshape(Cg.shape)
        ax.contourf(Cg,Hg,Zg,levels=10,cmap='Greys',alpha=0.3)
        ax.contour(Cg,Hg,Zg,levels=5,colors='gray',linewidths=0.5,alpha=0.5,linestyles='dashed')
    except: pass
    # Data circles at ACTUAL coordinates
    col=grid_cols[pi]; cmap=grid_cmaps[pi]; vmin=grid_vmins[pi]; vmax=grid_vmaxs[pi]
    if pi==0: scatter_norm=plt.Normalize(vmin=vmin,vmax=vmax)
    elif pi==1: scatter_norm=TwoSlopeNorm(vcenter=5,vmin=vmin,vmax=vmax)
    else: scatter_norm=TwoSlopeNorm(vcenter=0,vmin=vmin,vmax=vmax)
    valid_rows = grid_df[grid_df[col].notna()]
    if len(valid_rows)>0:
        ax.scatter(valid_rows['charge_actual'],valid_rows['hydrophob_actual'],
                  c=valid_rows[col],s=20,cmap=cmap,norm=scatter_norm,
                  edgecolors='black',linewidths=0.3,alpha=0.9,zorder=10)
    # X markers at TARGET coordinates
    ax.scatter(grid_df['ct'],grid_df['ht'],s=22,marker='x',c=x_marker_colors[pi],
              linewidths=0.6,alpha=0.7,label='Target',zorder=5)
    ax.set_xlabel('Charge',fontsize=6,labelpad=1)
    ax.set_ylabel('Hydrophobicity',fontsize=6,labelpad=1) if pi==0 else None
    ax.set_xlim(-2.5,13.5); ax.set_ylim(-1,1)
    ax.grid(True,alpha=0.25,linestyle='--',linewidth=0.3)
    ax.tick_params(labelsize=6,length=2)
    for spine in ax.spines.values(): spine.set_edgecolor('grey'); spine.set_linewidth(0.5)
    ax.legend(fontsize=6,loc='upper left',framealpha=0.8,edgecolor='gray')
    # Colorbar and legend
    if pi==0:
        sm=plt.cm.ScalarMappable(cmap=cmap,norm=plt.Normalize(vmin=vmin,vmax=vmax))
    elif pi==1:
        sm=plt.cm.ScalarMappable(cmap=cmap,norm=TwoSlopeNorm(vcenter=5,vmin=vmin,vmax=vmax))
    else:
        sm=plt.cm.ScalarMappable(cmap=cmap,norm=TwoSlopeNorm(vcenter=0,vmin=vmin,vmax=vmax))
    divider=make_axes_locatable(ax)
    cax=divider.append_axes("right",size="5%",pad=0.05)
    cb=plt.colorbar(sm,cax=cax)
    cb.set_label(grid_labels[pi],fontsize=6); cb.ax.tick_params(labelsize=6)

# -- D: Benchmarking UpSet --
ax_main=fig.add_subplot(gs_d[0]); ax_mat=fig.add_subplot(gs_d[1]); ax_leg=fig.add_subplot(gs_d[2])
ax_main.text(-0.12,1.12,'D',transform=ax_main.transAxes,**PL)
mc={'AMP-GAN':'#DDAA33','AMP-Diffusion':'#BB5566','Diff-AMP':'#004488','CPL-Diff':'#997700',
    'HydrAMP':'#EE8866','AMP-LM':'#6699CC','AMP-Muller':'#994455','OmegAMP-U':'#88CCAA','OmegAMP-T':'#117766','OmegAMP-P':'#44AA99'}
inters=['C','L','H','C+L','C+H','L+H','C+L+H']; props=['C','L','H']
xp=np.arange(len(inters))
for sep in [2.5,5.5]: ax_main.axvline(sep,color='#ddd',lw=0.6,zorder=0)
for model,data in bench_results.items():
    yv=[data.get(i,0) for i in inters]; mk='D' if model.startswith('OmegAMP') else 'o'
    al=1.0 if model.startswith('OmegAMP') else 0.6
    ax_main.scatter(xp,yv,c=mc.get(model,'#999'),s=12,marker=mk,alpha=al,edgecolors='white',linewidths=0.3,zorder=3,label=model)
ax_main.set_ylabel('Cond. Accuracy (%)',fontsize=6); ax_main.set_ylim(25,102); ax_main.set_xlim(-0.5,6.5); ax_main.set_xticks([])
ax_main.tick_params(axis='y',labelsize=6); ax_main.yaxis.grid(True,alpha=0.2,ls='--'); ax_main.spines['bottom'].set_visible(False)
for sep in [2.5,5.5]: ax_mat.axvline(sep,color='#ddd',lw=0.6,zorder=0)
for i,inter in enumerate(inters):
    inc=inter.split('+')
    for j,prop in enumerate(props):
        yp=2-j
        if prop in inc: ax_mat.scatter(i,yp,s=5,c='#333',zorder=3)
        else: ax_mat.scatter(i,yp,s=5,facecolors='white',edgecolors='#ccc',linewidths=0.5,zorder=3)
    if '+' in inter:
        idx=[2-props.index(p) for p in inc]; ax_mat.plot([i,i],[min(idx),max(idx)],c='#333',lw=0.8,zorder=2)
ax_mat.set_yticks(range(3)); ax_mat.set_yticklabels(props[::-1],fontsize=6)
ax_mat.set_xlim(-0.5,6.5); ax_mat.set_ylim(-0.5,2.5); ax_mat.set_xticks([])
ax_mat.tick_params(left=False,bottom=False)
for sp in ax_mat.spines.values(): sp.set_edgecolor('#ddd')
ax_mat.spines['top'].set_visible(False)
ax_leg.axis('off')
h,l=ax_main.get_legend_handles_labels()
ax_leg.legend(h,l,loc='center',fontsize=6,frameon=False,ncol=4,columnspacing=0.4,handletextpad=0.2,markerscale=0.6)

# -- E: Property distributions (matching original styling exactly) --
FC = {'amp':'#EE7733','peptides':'#0077BB','unconditional':'#888888','targeted':'#009988','prototype':'#CC3311','target':'#228B22'}
XL=(-4,12); YL=(-1.1,1.0)

def compute_kde(x, y, xlim, ylim):
    mask = (x>=xlim[0])&(x<=xlim[1])&(y>=ylim[0])&(y<=ylim[1])
    xi, yi = np.mgrid[xlim[0]:xlim[1]:100j, ylim[0]:ylim[1]:100j]
    kde = gaussian_kde(np.vstack([x[mask], y[mask]]))
    zi = kde(np.vstack([xi.flatten(), yi.flatten()])).reshape(xi.shape)
    return xi, yi, zi

np.random.seed(42)
sub_amp = amp_props.sample(min(5000,len(amp_props)),random_state=42)
sub_pep = pep_props.sample(min(5000,len(pep_props)),random_state=42)
sub_unc = unc_props.sample(min(5000,len(unc_props)),random_state=42)
sub_tar = tar_props.sample(min(5000,len(tar_props)),random_state=42)
sub_pop = pop_props.sample(min(5000,len(pop_props)),random_state=42)

# Left panel: AMPs + Peptipedia + OmegAMP-U
ax=fig.add_subplot(gs_e[0])
ax.text(-0.08,1.12,'E',transform=ax.transAxes,**PL)
ax.set_xlim(XL); ax.set_ylim(YL)
xi,yi,zi = compute_kde(sub_amp['charge'].values, sub_amp['hydrophobicity'].values, XL, YL)
ax.contour(xi,yi,zi,levels=[zi.max()*p for p in [0.1,0.4]],colors=FC['amp'],linestyles='--',linewidths=[0.5,0.9],alpha=0.9)
xi,yi,zi = compute_kde(sub_pep['charge'].values, sub_pep['hydrophobicity'].values, XL, YL)
ax.contour(xi,yi,zi,levels=[zi.max()*p for p in [0.1,0.4]],colors=FC['peptides'],linestyles='--',linewidths=[0.5,0.9],alpha=0.9)
xi,yi,zi = compute_kde(sub_unc['charge'].values, sub_unc['hydrophobicity'].values, XL, YL)
ax.contour(xi,yi,zi,levels=[zi.max()*p for p in [0.1,0.5]],colors=FC['unconditional'],linestyles='-',linewidths=[0.5,0.9],alpha=0.8)
ax.set_xlabel('Charge',fontsize=6); ax.set_ylabel('Hydrophobicity',fontsize=6)
ax.tick_params(labelsize=6,length=2,pad=1)

# Right panel: OmegAMP-U + OmegAMP-T + OmegAMP-P + Target
ax=fig.add_subplot(gs_e[1])
ax.set_xlim(XL); ax.set_ylim(YL)
tr = {'charge':(2,10),'hydrophobicity':(-0.5,0.8)}
ax.add_patch(Rectangle((tr['charge'][0],tr['hydrophobicity'][0]),tr['charge'][1]-tr['charge'][0],
                       tr['hydrophobicity'][1]-tr['hydrophobicity'][0],linewidth=0.6,
                       edgecolor=FC['target'],facecolor=FC['target'],alpha=0.12,zorder=0))
xi,yi,zi = compute_kde(sub_unc['charge'].values, sub_unc['hydrophobicity'].values, XL, YL)
ax.contour(xi,yi,zi,levels=[zi.max()*p for p in [0.1,0.5]],colors=FC['unconditional'],linestyles='-',linewidths=[0.5,0.9],alpha=0.8)
xi,yi,zi = compute_kde(sub_tar['charge'].values, sub_tar['hydrophobicity'].values, XL, YL)
ax.contour(xi,yi,zi,levels=[zi.max()*p for p in [0.1,0.5]],colors=FC['targeted'],linestyles='-',linewidths=[0.6,1.0],alpha=0.95)
xi,yi,zi = compute_kde(sub_pop['charge'].values, sub_pop['hydrophobicity'].values, XL, YL)
ax.contour(xi,yi,zi,levels=[zi.max()*p for p in [0.1,0.5]],colors=FC['prototype'],linestyles='-',linewidths=[0.6,1.0],alpha=0.95)
ax.set_xlabel('Charge',fontsize=6)
ax.tick_params(labelsize=6,length=2,pad=1); ax.set_yticklabels([])

# Shared legend
legend_elements = [
    Line2D([0],[0],color=FC['amp'],linestyle='--',linewidth=1,label='AMPs'),
    Line2D([0],[0],color=FC['peptides'],linestyle='--',linewidth=1,label='Peptipedia'),
    Line2D([0],[0],color=FC['unconditional'],linestyle='-',linewidth=1,label='OmegAMP-U'),
    Line2D([0],[0],color=FC['targeted'],linestyle='-',linewidth=1,label='OmegAMP-T'),
    Line2D([0],[0],color=FC['prototype'],linestyle='-',linewidth=1,label='OmegAMP-P'),
    Patch(facecolor=FC['target'],alpha=0.3,edgecolor=FC['target'],linewidth=0.6,label='Target'),
]
fig.legend(handles=legend_elements,loc='lower center',bbox_to_anchor=(0.76,0.17),ncol=3,fontsize=6,
          frameon=False,handletextpad=0.4,handlelength=1.2,columnspacing=1.0)

plt.savefig('../figures/figure2_conditioning.pdf',bbox_inches='tight')
plt.savefig('../figures/figure2_conditioning.png',bbox_inches='tight')
plt.show()
print("Saved figure2_conditioning")